# Mobile Suit AMBAC Wireframe Demo

This notebook demonstrates:
- Loading a minimal wireframe mobile suit model in MuJoCo
- Parameterizing thruster positions
- Running a simulation with thruster control
- Rendering frames and creating a GIF

## Important: Run cells in order!
1. **Run cell 1 first** - Installs OSMesa for headless rendering
2. If you get OpenGL errors, **Restart runtime** and run cell 1 first

In [ ]:
# ============================================
# COLAB SETUP - Run this cell FIRST!
# ============================================
# IMPORTANT: If you get OpenGL errors, restart the runtime 
# (Runtime > Restart runtime) and run this cell first!

import os
import sys

# Check if mujoco already imported (would cause issues)
if 'mujoco' in sys.modules:
    print("WARNING: mujoco already imported!")
    print("Please restart runtime (Runtime > Restart runtime) and run this cell first.")
else:
    # Set rendering backend BEFORE any mujoco import
    os.environ['MUJOCO_GL'] = 'osmesa'
    print("Set MUJOCO_GL=osmesa")

# 1. Install dependencies
!pip install -q mujoco dm-control imageio imageio[ffmpeg] matplotlib

# 2. Install OSMesa for headless rendering
!apt-get install -qq libosmesa6-dev libgl1-mesa-glx libglfw3 patchelf > /dev/null 2>&1

# 3. Clone repo
!git clone https://github.com/OsirisRaptor/mobile-suit-sim-mujoco.git 2>/dev/null || echo "Repo already cloned"
%cd mobile-suit-sim-mujoco

print("\nSetup complete! Now run the next cells.")

In [ ]:
import os
import numpy as np
import mujoco
from mobile_suit_sim.config import ThrusterConfig
from mobile_suit_sim.model_builder import load_model, apply_thruster_config
from mobile_suit_sim.sim_runner import simulate_and_render
from IPython.display import Image, display

# Path to MJCF
mjcf_path = os.path.join("models", "ambac_test.mjcf")

model, data = load_model(mjcf_path)

print("nq (positions):", model.nq)
print("nv (velocities):", model.nv)
print("nbody:", model.nbody)
print("nsite:", model.nsite)

In [ ]:
# Define a single thruster config
thruster_cfgs = [
    ThrusterConfig(
        name="main",          # maps to site "thruster_main"
        body_name="torso",    # not used yet, but good for future checks
        pos=[0.0, -0.6, 0.0], # x, y, z in torso frame
        dir_world=[0.0, -1.0, 0.0],
        max_force=2000.0
    )
]

site_ids = apply_thruster_config(model, thruster_cfgs)
site_ids

In [ ]:
def control_fn(step, model, data):
    # We only have one general actuator: thruster_main_act
    # In general actuators, data.ctrl is in the same order as actuators.
    n_steps_burn = 150

    if step < n_steps_burn:
        data.ctrl[:] = 1500.0  # N, < ctrlrange max
    else:
        data.ctrl[:] = 0.0

In [ ]:
output_gif = simulate_and_render(
    model=model,
    data=data,
    n_steps=300,
    control_fn=control_fn,
    width=480,
    height=360,
    camera="fixed",          # or default
    output_path="ambac_thruster_demo.gif"
)

display(Image(filename=output_gif))